# Project — Airline AI Assistant (Multi-modal)

**Reference guide:** take the Day 3 chat UI and Day 4 tool loop, then add **image generation** and **speech** in a custom Gradio `Blocks` app.

Day 4 proved the model cannot be trusted with fares unless *your* Python looks them up. Day 5 keeps that handshake and layers two more models on the same turn: a destination picture (`gpt-image-1-mini`) and a spoken reply (`gpt-4o-mini-tts`). The LLM still only *talks*; your functions still *do*.

Course source: `week2/day5.ipynb`. This notebook is the annotated, logic-corrected version for reuse.

### What you'll learn

1. **Standalone FlightAI + SQLite** — same `get_ticket_price` tool as Day 4, bootstrapped so this notebook runs even if `prices.db` is missing
2. **Baseline vs tools** — `ChatInterface` without tools (guesses) then with the Day 4 `while` loop (real fares)
3. **Image generation** — `artist(city)` decodes `b64_json` into a PIL `Image`
4. **Text-to-speech** — `talker(text)` returns MP3 bytes Gradio can play
5. **`gr.Blocks`** — custom layout; chained `.submit(...).then(...)` so the user message appears before the slow image/TTS work
6. **Agent-shaped turn** — one user question → tool(s) → spoken answer → optional destination image

> Run cells top-to-bottom. Later cells redefine `chat`. Each `launch()` starts another local Gradio app (ports increment: 7860, 7861, …). Image generation costs about **$0.03 per picture** — skip the `artist` demo cell if you are iterating on chat logic.

### Code review (what this notebook originally had wrong)

| Issue | Why it matters | Fix in this notebook |
|---|---|---|
| `get_ticket_price` called `.lower()` on a missing city | `arguments.get("destination_city")` can be `None` → `AttributeError` in the UI | Default to `""`, reject empty cities |
| `handle_tool_calls_and_return_cities` skipped unknown names | The API requires a `role: tool` result for **every** `tool_call_id` | Fallback `"Unknown tool: …"` |
| `cities = new_cities` inside the `while` | A second tool round **replaced** cities from the first round, so the image could be for the wrong city (or none) | `cities.extend(...)` |
| Helpers defined *after* the `chat` that calls them | Works only because Python binds names at call time; easy to launch the UI too early | Define handlers first |
| `talker(reply)` when `content` is `None` | TTS rejects empty/None input | Coerce to `""` and skip empty audio |
| `pop_art` in the image prompt | Underscore is not a style name; the model draws a weaker poster | `pop-art` (course wording) |
| `prices.db` assumed to exist | This folder only has a DB if Day 4 ran | `CREATE TABLE IF NOT EXISTS` + seed if empty |
| `auth=("michael", "123456")` on the final launch | A demo password baked into a reference notebook | Optional; commented, same pattern as Day 2 |
| No loop cap | A confused model can request tools forever | `MAX_TOOL_ROUNDS` from Day 4 |


## 1. Setup

Same stack as Days 3–4: env key, one OpenAI client, Gradio, SQLite.

`json` is required because tool arguments arrive as a **JSON string**. `base64` / `BytesIO` / `PIL` arrive later for images; they are imported in that section so the chat-only path does not need them.


In [ ]:
# --- Imports ---
# os          : read OPENAI_API_KEY from the environment
# json        : parse tool arguments the model returns as a JSON string
# sqlite3     : ticket catalog (created in Day 4; bootstrapped here if missing)
# load_dotenv : load a local .env file into os.environ
# OpenAI      : Chat Completions + Images + TTS on the same client
# gradio      : ChatInterface (recap) then Blocks (final UI)
import json
import os
import sqlite3
from typing import Any, TypedDict

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


class ChatMessage(TypedDict):
    """One turn in the OpenAI / Gradio messages format."""
    role: str      # "system" | "user" | "assistant" | "tool"
    content: str


load_dotenv(override=True)

openai_api_key: str | None = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — add OPENAI_API_KEY to your .env file")


### Client, model, and database path

`OpenAI()` reads `OPENAI_API_KEY`. Chat, images, and TTS all go through this one client — different methods (`chat.completions`, `images.generate`, `audio.speech`), same credentials.

`MODEL` is the **chat** model (must support function tools). Image and speech models are hard-coded in `artist` / `talker` because they are different products.

`DB` is a relative path: Jupyter's working directory should be this `Week 2` folder so `prices.db` is the file Day 4 wrote.


In [ ]:
openai = OpenAI()

# Chat model for the assistant. Day 4 used gpt-5.4-nano; the course Day 5 notebook
# uses gpt-4.1-mini. Either is fine if it supports function tools.
MODEL = "gpt-4.1-mini"

# Local alternative (Ollama must already be running, and the model must support tools):
# MODEL = "llama3.2"
# openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

DB = "prices.db"
MAX_TOOL_ROUNDS = 5  # safety cap for the tool while-loop (same as Day 4)

print(f"Using chat model: {MODEL}")
print(f"Ticket database: {DB}")


## 2. System message (airline persona)

Unchanged from Days 3–4: short, courteous, honest. Short answers make a **guessed** fare obvious in the no-tools baseline, and they keep TTS clips brief.

We are not adding a write-tool prompt here (that was the Day 4 exercise). This assistant only **reads** prices.


In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
""".strip()


## 3. SQLite lookup (source of truth)

Day 4 replaced an in-memory dict with `prices.db`. This cell makes the notebook **runnable on its own**: create the table if needed, seed the course catalog only when the table is empty (so a Day 4 write-tool experiment is not overwritten).

- Parameterized `?` placeholders — the city string comes from the model; never interpolate it into SQL
- `.lower()` so `"London"` and `"london"` hit the same row
- Empty / missing city returns a clear string instead of crashing on `.lower()`


In [ ]:
SEED_PRICES = {
    "london": 799.0,
    "paris": 899.0,
    "tokyo": 1420.0,
    "berlin": 499.0,
    "sydney": 2999.0,
}


def ensure_prices_db(path: str = DB) -> None:
    """Create prices.db if missing; seed only when the table has no rows."""
    with sqlite3.connect(path) as conn:
        conn.execute(
            "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
        )
        count = conn.execute("SELECT COUNT(*) FROM prices").fetchone()[0]
        if count == 0:
            conn.executemany(
                "INSERT INTO prices (city, price) VALUES (?, ?)",
                list(SEED_PRICES.items()),
            )
            print(f"Seeded {path} with {len(SEED_PRICES)} cities")
        else:
            print(f"{path} already has {count} row(s) — leaving data as-is")
        conn.commit()


def get_ticket_price(destination_city: str) -> str:
    """Look up a return-ticket price. Safe if the city is missing or unknown."""
    city = (destination_city or "").strip().lower()
    print(f"DATABASE TOOL CALLED: Getting price for {city or '<empty>'}", flush=True)
    if not city:
        return "No destination city was provided"

    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city,),
        ).fetchone()

    if row is None:
        return f"No price data is available for {destination_city}"
    return f"Ticket price to {destination_city} is ${row[0]:.2f}"


ensure_prices_db()


Smoke-test the function **before** the LLM is involved. If this cell fails, the chatbot will fail the same way — but with a Gradio stack trace instead of a clear error.


In [ ]:
print(get_ticket_price("london"))  # expected: $799.00
print(get_ticket_price("Paris"))   # case-insensitive
print(get_ticket_price("Mars"))    # unknown city — must not crash or invent a fare
print(get_ticket_price(""))        # missing argument path


## 4. Describe the function to the model (JSON Schema)

This dict is **not** Python. It is a JSON Schema the API forwards to the model (Day 4 §6).

| Field | Role |
|---|---|
| `name` | Must match the Python function you dispatch on |
| `description` | The model uses this to decide *whether* to call the tool |
| `parameters.properties` | Argument names and types the model must fill in |
| `required` | Arguments that must be present |
| `additionalProperties: False` | Reject extra invented keys |

Then wrap it: `tools = [{"type": "function", "function": price_function}]`.


In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": (
        "Get the price of a return ticket to the destination city. "
        "Use this whenever the customer asks what a flight costs."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": price_function}]
tools


## 5. Baseline chatbot — no tools (Day 3 recap)

Same message assembly as Day 3 / Day 4 §3:

```text
messages = [system] + history + [latest user turn]
```

**Try this in the UI:** "How much is a return ticket to London?"

The model has **no fare table**. It may refuse (good system prompt) or invent `$499`. Either way you cannot *trust* the number. The next section replaces guessing with a lookup.


In [ ]:
def project_history(history: list[dict[str, Any]]) -> list[ChatMessage]:
    """Keep only role + content. Gradio may attach extra keys we must not send."""
    return [{"role": h["role"], "content": h["content"]} for h in history]


def chat_no_tools(message: str, history: list[dict[str, Any]]) -> str | None:
    messages: list[dict[str, Any]] = (
        [{"role": "system", "content": system_message}]
        + project_history(history)
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


gr.ChatInterface(fn=chat_no_tools, type="messages").launch()


## 6. Tool loop — real fares (Day 4 recap)

Frontier models can emit a structured **tool call** instead of a user-facing sentence.

```text
system
...prior history...
user          "How much to London?"
assistant     content=None, tool_calls=[{id, name=get_ticket_price, arguments=...}]
tool          tool_call_id=<same id>, content="Ticket price to London is $799.00"
assistant     "A return ticket to London is $799."
```

Rules carried over from Day 4:

1. Append the **assistant tool-call message object** the API returned (not a homemade dict).
2. Append one `role: "tool"` message per `tool_call_id` — including unknown names.
3. Keep offering `tools=tools` inside the `while` so the model can request a follow-up.
4. Do not reuse the parameter name `message` for `response.choices[0].message`.
5. Cap the loop (`MAX_TOOL_ROUNDS`).

`handle_tool_calls` also collects destination cities. The ChatInterface path ignores that list; the multi-modal `Blocks` path uses it to decide what to draw.


In [ ]:
def handle_tool_calls(
    assistant_msg: Any,
) -> tuple[list[dict[str, str]], list[str]]:
    """Run every tool the model requested.

    Returns
    -------
    responses : list of role=tool messages (one per tool_call_id)
    cities    : destination_city values from get_ticket_price calls
    """
    responses: list[dict[str, str]] = []
    cities: list[str] = []

    for tool_call in assistant_msg.tool_calls or []:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments or "{}")

        if name == "get_ticket_price":
            city = arguments.get("destination_city") or ""
            if city:
                cities.append(city)
            content = get_ticket_price(city)
        else:
            # Still must reply — the API pairs each tool_call_id with a result.
            content = f"Unknown tool: {name}"

        responses.append(
            {
                "role": "tool",
                "content": content,
                "tool_call_id": tool_call.id,
            }
        )
    return responses, cities


def chat(message: str, history: list[dict[str, Any]]) -> str | None:
    """ChatInterface callback: (message, history) -> assistant text."""
    messages: list[Any] = (
        [{"role": "system", "content": system_message}]
        + project_history(history)
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    rounds = 0
    while response.choices[0].finish_reason == "tool_calls":
        rounds += 1
        if rounds > MAX_TOOL_ROUNDS:
            break
        assistant_msg = response.choices[0].message
        tool_results, _cities = handle_tool_calls(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(tool_results)
        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    return response.choices[0].message.content


**Prompts to try** (watch the notebook stdout for `DATABASE TOOL CALLED`):

- "How much is a return to London?" → one tool call, then a short sentence with `$799.00`
- "Compare London and Tokyo" → two tool calls in one turn
- "How much to Mars?" → tool runs, returns unknown, assistant should not invent a fare
- "What's your baggage policy?" → no tool; the model should admit it does not know

If `DATABASE TOOL CALLED` never prints, the model answered from weights alone (forgot `tools=`, or the model does not support function tools).


In [ ]:
gr.ChatInterface(fn=chat, type="messages", title="FlightAI (tools)").launch()


### What Gradio is actually doing

1. It builds a frontend (Svelte) from your Python UI description.
2. It starts a Starlette server on a free port and serves that app.
3. It exposes backend routes for your callbacks (`chat`, later `put_message_in_chatbot`).

The Submit control is wired to those routes. That is the whole trick: Python functions become HTTP handlers, and the generated UI calls them.

Day 2 covered `gr.Interface`. Day 3 covered `gr.ChatInterface`. The final app here needs **`gr.Blocks`** because we want a chatbot, an image, and an audio player on one page — ChatInterface has no slot for those extras.


## 7. Image generation — `artist`

`openai.images.generate` is a different endpoint from chat. There is no conversation history and no tools. You send a prompt; you get pixels.

This course uses `gpt-image-1-mini` (the image model behind GPT-5-class ChatGPT). The response field we use is `b64_json`: a base64-encoded PNG/JPEG sitting in JSON, not a URL. Decode → `BytesIO` → `PIL.Image`. Gradio's `gr.Image` accepts that PIL object directly.

### Price alert

Each image is on the order of **3 cents**. Do not put `artist` in a tight loop. The final app generates **at most one** image per user turn (the first destination city), and only if a price tool actually ran.


In [ ]:
# Image pipeline only — chat-only cells above do not need these.
import base64
from io import BytesIO

from PIL import Image


In [ ]:
IMAGE_MODEL = "gpt-image-1-mini"


def artist(city: str) -> Image.Image | None:
    """Generate a pop-art vacation poster for a destination city.

    Returns None if the city is empty or the API omitted image data, so the
    Gradio Image component can stay blank instead of crashing the turn.
    """
    city = (city or "").strip()
    if not city:
        return None

    image_response = openai.images.generate(
        model=IMAGE_MODEL,
        prompt=(
            f"An image representing a vacation in {city}, showing tourist spots "
            f"and everything unique about {city}, in a vibrant pop-art style"
        ),
        size="1024x1024",
        n=1,
    )
    payload = image_response.data[0] if image_response.data else None
    if payload is None or not payload.b64_json:
        print("Image API returned no b64_json")
        return None

    image_bytes = base64.b64decode(payload.b64_json)
    return Image.open(BytesIO(image_bytes))


Optional smoke test. **Skip this cell** while you iterate on chat — it spends money and a few seconds. Uncomment when you want to confirm the image path.


In [ ]:
# OPTIONAL (~$0.03 and several seconds). Uncomment to test:
# image = artist("New York City")
# display(image)
print("artist() is defined. Uncomment the lines above to generate a test image.")


## 8. Text-to-speech — `talker`

`openai.audio.speech.create` turns a string into audio bytes (MP3 by default).

Two easy mistakes:

1. **Notebook display.** Returning `bytes` prints `b'\xff\xf3...'`. Wrap with `IPython.display.Audio` (or pass `play=True`) to hear it here. Gradio's `gr.Audio` wants the raw bytes later — so `talker` **returns bytes**, and playing in the notebook is opt-in.
2. **Empty text.** If the chat model returns `content=None`, TTS will error. Guard before calling the API.


In [ ]:
from IPython.display import Audio, display

TTS_MODEL = "gpt-4o-mini-tts"
TTS_VOICE = "onyx"  # also try: alloy, coral, verse, sage


def talker(message: str, play: bool = False) -> bytes | None:
    """Turn text into MP3 bytes. Gradio Audio consumes the return value.

    Return None (not b'') when there is nothing to speak — empty bytes can
    make gr.Audio error.
    """
    text = (message or "").strip()
    if not text:
        print("talker: empty text, skipping TTS")
        return None

    response = openai.audio.speech.create(
        model=TTS_MODEL,
        voice=TTS_VOICE,
        input=text,
        response_format="mp3",
    )
    audio_bytes = response.content
    print(f"Generated {len(audio_bytes):,} bytes of MP3")
    if play:
        display(Audio(audio_bytes, autoplay=True))
    return audio_bytes


In [ ]:
# play=True shows an audio player under this cell (click play if autoplay is blocked)
audio = talker("A return ticket to London is $799.", play=True)


## 9. One agent-shaped turn

Wire the three models into a single callback:

1. **Chat + tools** — same `while` loop as §6; accumulate cities across rounds
2. **TTS** — speak the final assistant sentence
3. **Image** — if any price lookup happened, draw the **first** city (cost cap)

### Why `chat(history)` instead of `chat(message, history)`?

`gr.ChatInterface` appends the user turn for you and passes `(message, history)`.

`gr.Blocks` does not. We split the work:

| Step | Function | Why |
|---|---|---|
| 1 | `put_message_in_chatbot` | Clear the textbox; append `{role: user}` so the UI updates **immediately** |
| 2 | `chat_multimodal` | Runs tools + TTS + image (slow). History already contains the user turn |

If `chat_multimodal` also appended the user message, every question would appear twice.


In [ ]:
def chat_multimodal(
    history: list[dict[str, Any]],
) -> tuple[list[dict[str, Any]], bytes | None, Image.Image | None]:
    """Blocks callback: history already includes the latest user turn.

    Returns (updated history, mp3 bytes or None, optional PIL image) to match
    the three output components in the UI cell below.
    """
    messages: list[Any] = (
        [{"role": "system", "content": system_message}]
        + project_history(history)
    )
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

    cities: list[str] = []
    rounds = 0
    while response.choices[0].finish_reason == "tool_calls":
        rounds += 1
        if rounds > MAX_TOOL_ROUNDS:
            break
        assistant_msg = response.choices[0].message
        tool_results, new_cities = handle_tool_calls(assistant_msg)
        cities.extend(new_cities)  # accumulate — do not replace
        messages.append(assistant_msg)
        messages.extend(tool_results)
        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    reply = response.choices[0].message.content or ""
    history = history + [{"role": "assistant", "content": reply}]

    voice = talker(reply)
    image = artist(cities[0]) if cities else None
    return history, voice, image


## 10. Custom UI with `gr.Blocks`

The three Gradio entry points, from least to most control (Day 2 / Day 3 / here):

| Constructor | Use when |
|---|---|
| `gr.Interface` | One function, labeled inputs/outputs (brochure generator, Day 2) |
| `gr.ChatInterface` | Standard chatbot; it owns history for you (Days 3–4) |
| `gr.Blocks` | You place components and wire events yourself |

### Event chain

```text
textbox.submit
    → put_message_in_chatbot(message, history)
         outputs: cleared textbox, chatbot with user bubble
    → .then(chat_multimodal, inputs=chatbot)
         outputs: chatbot with assistant bubble, audio, image
```

`.then` runs after the previous step finishes and uses the **updated** chatbot state. That is why `chat_multimodal` receives history that already includes the user message.


In [ ]:
def put_message_in_chatbot(
    message: str, history: list[dict[str, Any]]
) -> tuple[str, list[dict[str, Any]]]:
    """Optimistic UI: show the user bubble before the slow agent turn."""
    return "", history + [{"role": "user", "content": message}]


with gr.Blocks(title="FlightAI Multi-modal") as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(
            label="Chat with our AI Assistant:",
            placeholder="How much is a return to London?",
        )

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(
        chat_multimodal,
        inputs=chatbot,
        outputs=[chatbot, audio_output, image_output],
    )

# Day 2 pattern: pass auth=("user", "pass") for a demo login gate.
# Credentials would live in this notebook — leave it off for local reuse.
ui.launch(inbrowser=True)


**Prompts to try in the Blocks app**

- "How much is a return to London?" → spoken `$799.00` + London poster
- "Compare Paris and Tokyo" → both prices in one sentence; image is Paris (first city)
- "What's the weather in Berlin?" → no price tool, so **no new image** and a one-sentence refusal
- "How much to Mars?" → tool runs, unknown fare, no invented number; `artist("Mars")` still runs because a city was requested — skip-worthy if you are saving image budget

Stdout should show `DATABASE TOOL CALLED` then `Generated N bytes of MP3`. Image generation is the slow, expensive step.


## Quick reference (cheat sheet)

| Piece | Role |
|---|---|
| `system_message` | Persona + honesty; keep answers short for TTS |
| `get_ticket_price` | Real Python; the only code that reads `prices.db` |
| `price_function` / `tools` | JSON Schema the **chat** model sees |
| `handle_tool_calls` | Dispatch + `role: tool` messages + city list |
| `chat` | `ChatInterface` callback `(message, history) -> str` |
| `chat_multimodal` | `Blocks` callback `history -> (history, audio, image)` |
| `put_message_in_chatbot` | Immediate user bubble; do not duplicate this in `chat_multimodal` |
| `artist` | `images.generate` → `b64_json` → PIL |
| `talker` | `audio.speech.create` → MP3 bytes |
| `MAX_TOOL_ROUNDS` | Cap on sequential tool rounds |
| `.submit(...).then(...)` | Chain Gradio events; later step sees earlier outputs |

### Message protocol (unchanged from Day 4)

```python
response = openai.chat.completions.create(
    model=MODEL,
    messages=messages,   # system + history [+ assistant tool calls + tool results]
    tools=tools,
)
```

### Logic pitfalls to remember

1. **Two `chat` signatures** — ChatInterface wants `(message, history)`; Blocks here wants `(history,)` because the user turn was already appended.
2. **Replacing `cities` in the `while`** — extend the list or the poster city is wrong after a second tool round.
3. **Missing `role: tool` for unknown names** — the follow-up `create` will fail.
4. **`None` city / `None` reply** — `.lower()` and TTS both blow up; coerce to `""`.
5. **Forgetting `tools=` inside the loop** — the model cannot request a second function.
6. **Image cost** — one `artist` call per turn, and only if a destination was looked up.
7. **SQL injection** — always bound parameters; the city is model-chosen text.
8. **Relative `prices.db`** — working directory must be this folder.

### How the three models share one turn

```text
user text
    → gpt-4.1-mini  (maybe tool_calls)
    → your SQLite
    → gpt-4.1-mini  (final sentence)
    → gpt-4o-mini-tts  (always, if there is text)
    → gpt-image-1-mini (only if a city was looked up)
```

The LLM never opens the database, never paints, and never speaks. Those are ordinary Python calls you scheduled after (or during) the chat handshake.


## Exercises and business applications

From the course notebook, plus a few that follow from the fixes above:

1. **More tools** — simulate booking a seat (community contributions under `week2/` have examples). Reuse `handle_tool_calls` / a dispatch table from Day 4 rather than growing `if/elif`.
2. **Your domain** — same Blocks shell: swap `get_ticket_price` for a CRM lookup, an onboarding checklist, or an inventory query. Keep image/TTS optional.
3. **Image policy** — only generate when the user asks for a picture, or cache by city so London is not redrawn every turn.
4. **Streaming** — Day 3 streamed tokens into ChatInterface. Blocks can stream into `gr.Chatbot` too; TTS still has to wait for the full sentence.

Next in the course: the Week 2 end-of-week exercise notebook.


## Conclusion

Day 5 is assembly, not a new protocol.

You already had a polite FlightAI (Day 3) and a tool loop over SQLite (Day 4). This notebook puts those behind a custom Gradio layout and spends two extra API calls per turn: one to **say** the answer, one to **show** the destination. The message protocol is identical — `finish_reason == "tool_calls"`, run Python, append `role: tool`, call again.

Reuse this notebook as a template: copy `handle_tool_calls` + `chat_multimodal` + the Blocks chain, replace the lookup and the prompts, keep the handshake. For production, gate image/TTS on user intent, treat tool arguments as untrusted input, and do not ship demo `auth=` passwords in source.
